# 01 — Text Basics
**Goal:** Dominar las operaciones fundamentales sobre texto antes de usar cualquier biblioteca de NLP.

NLP siempre empieza con texto crudo lleno de ruido. Este notebook cubre el pipeline de limpieza:
```
texto crudo → lowercase → regex clean → tokenización → normalización
```

Usaremos solo Python built-ins y `re` — sin NLTK ni spaCy.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import re
import string
import numpy as np
import pandas as pd
from collections import Counter

print('Python built-ins only — no NLP library needed for this notebook')

## 1. El problema con el texto crudo

In [ ]:
# Comentarios NPS reales se parecen a esto
raw_comments = [
    "La app está MUY LENTA!!! tardé 20 min en activar mi tarjeta 😤",
    "Excelente servicio!!! El proceso de solicitud fue rápido y fácil :)",
    "NO me llegó el OTP... llamé al call center y nadie responde.",
    "Muy buena experiencia, aprobaron mi crédito en < 5 minutos",
    "El proceso de carga de documentos falla TODO el tiempo #malServicio",
    "Recomiendo 100% la tarjeta, beneficios increíbles!!! 🎉🎉",
    "https://quejasclientes.com — esto es INACEPTABLE",
    "Llevo 3 días esperando la evaluación crediticia... sin respuesta.",
]

for c in raw_comments:
    print(repr(c))

## 2. Operaciones básicas de string

In [ ]:
text = "La App está MUY LENTA!!! tardé 20 min en activar mi tarjeta 😤"

print('lower:   ', text.lower())
print('strip:   ', '   hola   '.strip())
print('split:   ', text.lower().split()[:5])
print('replace: ', text.replace('!!!', '.'))
print('len:     ', len(text))
print('count:   ', text.lower().count('a'))
print('startswith:', text.startswith('La'))
print('in:      ', 'LENTA' in text)

## 3. Expresiones regulares — `re`

| Patrón | Qué captura |
|---|---|
| `\w` | alfanumérico + `_` |
| `\W` | NO alfanumérico |
| `\d` | dígito |
| `\s` | espacio/tab/newline |
| `+` | 1 o más |
| `*` | 0 o más |
| `[abc]` | cualquiera de a, b, c |
| `^...$` | inicio/fin de string |

In [ ]:
text = "La app está MUY LENTA!!! tardé 20 min 😤 https://queja.com #malServicio"

# Eliminar URLs
no_url = re.sub(r'https?://\S+', '', text)
print('sin URL:    ', no_url)

# Eliminar hashtags y emojis (todo lo que no sea letra, número o espacio)
clean = re.sub(r'[^\w\s]', ' ', no_url)      # puntuación → espacio
clean = re.sub(r'[^\x00-\x7Fáéíóúüñ\s]', '', clean)  # emojis out
print('sin emojis: ', clean)

# Colapsar espacios múltiples
clean = re.sub(r'\s+', ' ', clean).strip()
print('clean:      ', clean)

In [ ]:
# Extraer números mencionados (ej: tiempos de espera)
texts = [
    'tardé 20 minutos en activar',
    'Llevo 3 días esperando',
    'aprobaron en menos de 5 minutos',
    'llamé 4 veces al soporte',
]
for t in texts:
    nums = re.findall(r'\d+', t)
    print(f'{t:<45} → {nums}')

## 4. Tokenización — dividir en tokens

In [ ]:
def tokenize(text):
    """Limpieza básica + split en palabras."""
    text = text.lower()
    text = re.sub(r'https?://\S+', '', text)          # URLs
    text = re.sub(r'[^\x00-\x7Fáéíóúüñ\s]', '', text) # emojis
    text = re.sub(r'[^\wáéíóúüñ\s]', ' ', text)       # puntuación
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

for c in raw_comments[:4]:
    tokens = tokenize(c)
    print(f'  {tokens}')

## 5. Stopwords — palabras sin contenido semántico

In [ ]:
# Lista de stopwords en español (sin NLTK)
STOPWORDS_ES = set([
    'a','al','algo','algunas','algunos','ante','antes','como','con','contra',
    'cual','cuando','de','del','desde','donde','durante','e','el','ella',
    'ellas','ellos','en','entre','era','es','esa','esas','ese','eso','esos',
    'esta','estas','este','esto','estos','fue','han','has','hay','he','hasta',
    'la','las','le','les','lo','los','me','mi','mis','más','muy','no','nos',
    'o','para','pero','por','que','se','si','sin','sobre','su','sus','también',
    'tan','te','todo','todos','tu','tus','un','una','unas','unos','y','ya','yo',
])

def tokenize_clean(text, stopwords=STOPWORDS_ES, min_len=3):
    tokens = tokenize(text)
    return [t for t in tokens if t not in stopwords and len(t) >= min_len]

for c in raw_comments[:4]:
    print(tokenize_clean(c))

## 6. Análisis de frecuencia — qué palabras dominan

In [ ]:
# Cargar dataset sintético de feedback (generado en este curso)
import numpy as np
rng = np.random.default_rng(42)

templates = [
    # positivos
    'El proceso de solicitud fue muy rápido y eficiente',
    'Excelente servicio, aprobaron mi tarjeta rápidamente',
    'La app funciona perfecto, muy fácil de usar',
    'Increíbles beneficios, recomiendo la tarjeta al 100',
    'Atención al cliente excepcional, resolvieron todo rápido',
    'Proceso de activación muy sencillo y sin problemas',
    # negativos
    'La app está muy lenta, tardé mucho en activar la tarjeta',
    'No me llegó el OTP, tuve que llamar varias veces',
    'La carga de documentos falla constantemente',
    'Llevo días esperando la evaluación crediticia sin respuesta',
    'El call center no responde, pésimo servicio al cliente',
    'El proceso de aprobación es muy lento e ineficiente',
]

labels = [1]*6 + [0]*6
comments = [templates[rng.integers(0, 6 if i < 300 else 6, 1).item() + (0 if i < 300 else 6)] for i in range(600)]
sentiments = [1 if i < 300 else 0 for i in range(600)]

df_fb = pd.DataFrame({'comment': comments, 'sentiment': sentiments})
df_fb = df_fb.sample(frac=1, random_state=42).reset_index(drop=True)

# Frecuencia de palabras globales
all_tokens = []
for text in df_fb['comment']:
    all_tokens.extend(tokenize_clean(text))

freq = Counter(all_tokens)
print('Top 20 palabras:')
for word, count in freq.most_common(20):
    bar = '█' * (count // 10)
    print(f'  {word:20s} {count:4d}  {bar}')

In [ ]:
# Vocabulario único
vocab = sorted(set(all_tokens))
print(f'Vocabulario: {len(vocab)} palabras únicas')
print(f'Tokens totales: {len(all_tokens):,}')
print(f'Type-token ratio (diversidad léxica): {len(vocab)/len(all_tokens):.3f}')

## 7. N-gramas — frases como unidades

In [ ]:
def ngrams(tokens, n):
    return [' '.join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

text = 'proceso solicitud rápido fácil tarjeta'
tokens = text.split()

print('Unigramas:', ngrams(tokens, 1))
print('Bigramas: ', ngrams(tokens, 2))
print('Trigramas:', ngrams(tokens, 3))

# Bigramas más frecuentes en el dataset
all_bigrams = []
for text in df_fb['comment']:
    toks = tokenize_clean(text)
    all_bigrams.extend(ngrams(toks, 2))

print('\nTop bigramas:')
for bg, cnt in Counter(all_bigrams).most_common(10):
    print(f'  "{bg}" → {cnt}')

## Resumen
| Operación | Herramienta |
|---|---|
| Lowercase, strip, split | `str` built-ins |
| Eliminar URLs, emojis, puntuación | `re.sub` |
| Extraer patrones | `re.findall` |
| Tokenizar | `str.split()` post-limpieza |
| Filtrar stopwords | set lookup `O(1)` |
| Frecuencia de palabras | `collections.Counter` |
| N-gramas | ventana deslizante sobre lista de tokens |

**Siguiente:** `02_bag_of_words.ipynb` — representar texto como vectores numéricos.